In [ ]:
from utils import deserialize_circuits,random_circuits
import numpy as np
from qiskit import qpy
from qiskit.quantum_info import Operator
from matplotlib import pyplot as plt
from tqdm.notebook import trange

In [ ]:
# def calc_distance(target,qc):
#     generated = Operator(qc).to_matrix()
#     return np.linalg.norm(target-generated)**2/2
import numpy as np
from qiskit.quantum_info import Operator

# def calc_distance(target,qc):
#     generated = Operator(qc).to_matrix()
#     return np.linalg.norm(target-generated)**2/2

import numpy as np
from qiskit.quantum_info import Operator

def _to_unitary(obj):
    """Accepts np.ndarray / Operator / QuantumCircuit and returns a complex128 matrix."""
    if isinstance(obj, np.ndarray):
        U = obj
    else:
        U = Operator(obj).data
    return np.asarray(U, dtype=np.complex128)

def calc_process_fidelity(target, qc):
    """
    Phase-free process fidelity between target unitary and the circuit's unitary:
        F_pro = |Tr(U_target^† U_generated)|^2 / d^2   ∈ [0, 1]
    Invariant to global phase differences.
    """
    U_t = _to_unitary(target)
    U_g = _to_unitary(qc)

    if U_t.shape != U_g.shape:
        raise ValueError(f"Dimension mismatch: {U_t.shape} vs {U_g.shape}")

    d = U_t.shape[0]
    s = np.trace(U_t.conj().T @ U_g)          # complex scalar
    F_pro = (np.abs(s)**2) / (d * d)
    return float(np.clip(F_pro.real, 0.0, 1.0))

def calc_pf_loss(target, qc):
    """Loss version: 1 - process fidelity  (smaller is better)."""
    return 1.0 - calc_process_fidelity(target, qc)

def calc_distance(target, qc):
    return calc_pf_loss(target, qc)

In [ ]:
def benchmark(targets,circuits_collection):
    print("Benchmarking...")
    results = []
    for i in trange(len(targets)):
        target = targets[i]
        circuits = circuits_collection[i]
        # find the ciruit with the smallest distance
        min_distance = float('inf')
        min_circuit = None
        for qc in circuits:
            if len(qc.data)==0:
                continue
            distance = calc_distance(target,qc)
            if distance < min_distance:
                min_distance = distance
                min_circuit = qc
        results.append((min_distance,min_circuit))
    return results

In [ ]:
# First let's see how they perform on pretrained_dataset
targets_pretrained_file = "targets_param.npy"
targets = np.load(targets_pretrained_file)

circuits_collection_m_random = []
for i in trange(len(targets)):
    circuits_collection_m_random.append(random_circuits(1,3,3,12))
results_m_random = benchmark(targets,circuits_collection_m_random)

In [ ]:
print(results_m_random)

In [ ]:
import numpy as np
with open("distance_random_fixed2.txt", "w") as f:
    for idx, (value, _) in enumerate(results_m_random, start=1):
        f.write(f"{value}\n")